# LLM Intention Probing, Honesty, Deception, and Honest Mistakes, Algoverse 2026 Spring, KMSA & Tommy
## Part 1: Preparation

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")  # save plots to files only — do not display inline
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

# Settings — single source of truth for all paths, constants, and hyperparameters
from utils.settings import *

# Utils
from utils.knowledge_check import (
    knowledge_check_truthfulqa, knowledge_check_mmlu,
    run_knowledge_check_truthfulqa, run_knowledge_check_mmlu,
)
from utils.generation import (
    generate_response,
    run_factual_generation, run_scenario_generation,
    load_vllm_model,
    run_factual_generation_vllm, run_scenario_generation_vllm,
)
from utils.judge import (
    build_batch_requests_anthropic, parse_batch_results_anthropic,
    run_judge_anthropic,
    aggregate_judge_votes, build_full, print_threshold_summary,
)
from utils.activation import extract_activations, run_extract_activations, LABEL_MAP
from utils.analysis import (
    reduce_activations_pca, save_results_csv, select_pca_k, run_pca_reduction,
    filter_factual, build_probe_dataset, split_thinking_responses,
)
from utils.probe import (
    probe_all_layers, probe_all_layers_binary,
    probe_all_layers_cascaded, probe_all_layers_mlp, probe_all_layers_cascaded_mlp,
)
from utils.plotting import (
    plot_macro_f1, plot_perclass_f1, plot_auroc, plot_top_confusion_matrices,
)

# Reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Create output directories
for d in [
    KNOWLEDGE_TEST_DIR, RESPONSES_DIR,
    JUDGE_DIR, JUDGE_CLAUDE_HAIKU_DIR,
    OUTPUT_DIR, FIGURES_DIR,
    BINARY_DIR, TWAY_LR_DIR, TWAY_MLP_DIR, CASCADED_LR_DIR, CASCADED_MLP_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# Load fixed social scenario dataset
deception_df = pd.read_csv(DECEPTION_DATASET_PATH)
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts().to_string())

print(f"\nDevice: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_ID}")

deception_dataset: (400, 6)
label
honest       200
deceptive    200

Device: cuda
GPU:  NVIDIA GeForce RTX 4090
VRAM: 25.4 GB
Model: Qwen/Qwen3-4B


### 1.2 Load Model & Tokenizer

In [2]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     dtype=torch.bfloat16,
#     device_map="auto",
#     max_memory={0: "22GiB", "cpu": "120GiB"},
#     offload_folder="outputs/offload",
#     token=HF_READ_TOKEN
# )
# model.eval()

# # Support different config schemas (e.g., Gemma family and others).
# cfg = getattr(model.config, "text_config", model.config)
# N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
# HIDDEN_DIM = getattr(cfg, "hidden_size", getattr(cfg, "d_model", None))

# print(f"Loaded: {MODEL_ID}")
# print(f"Layers: {N_LAYERS}, hidden_dim: {HIDDEN_DIM}")

### 1.3 Load vLLM Model (alternative to 1.2 for generation — skip for activation extraction)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_READ_TOKEN)

from vllm import LLM
llm = LLM(model=MODEL_ID, dtype="bfloat16", gpu_memory_utilization=0.85, max_model_len=10052)

INFO 05-07 20:11:25 [config.py:717] This model supports multiple tasks: {'score', 'generate', 'embed', 'reward', 'classify'}. Defaulting to 'generate'.
INFO 05-07 20:11:25 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-07 20:11:31 [__init__.py:239] Automatically detected platform cuda.
INFO 05-07 20:11:34 [core.py:58] Initializing a V1 LLM engine (v0.8.5) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=10052, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='auto', reasoning_backend=None), observability_config=Observabil

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  3.80it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.66it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.83it/s]



INFO 05-07 20:11:36 [loader.py:458] Loading weights took 1.14 seconds
INFO 05-07 20:11:36 [gpu_model_runner.py:1347] Model loading took 7.5552 GiB and 1.583958 seconds
INFO 05-07 20:11:47 [backends.py:420] Using cache directory: /root/.cache/vllm/torch_compile_cache/c8b2e4215c/rank_0_0 for vLLM's torch.compile
INFO 05-07 20:11:47 [backends.py:430] Dynamo bytecode transform time: 10.21 s
INFO 05-07 20:11:51 [backends.py:136] Cache the graph of shape None for later use
INFO 05-07 20:12:26 [backends.py:148] Compiling a graph for general shape takes 38.67 s
INFO 05-07 20:13:00 [monitor.py:33] torch.compile takes 48.88 s in total
INFO 05-07 20:13:00 [kv_cache_utils.py:634] GPU KV cache size: 77,472 tokens
INFO 05-07 20:13:00 [kv_cache_utils.py:637] Maximum concurrency for 10,052 tokens per request: 7.71x
INFO 05-07 20:13:30 [gpu_model_runner.py:1686] Graph capturing finished in 29 secs, took 0.58 GiB
INFO 05-07 20:13:30 [core.py:159] init engine (profile, create kv cache, warmup model) took

## Part 2: Model Knowledge Test
### 2.1 TruthfulQA

In [5]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")

kc_tqa_df, tqa_passed_df, tqa_failed_df = run_knowledge_check_truthfulqa(
    tqa_mc, None, tokenizer, DEVICE, TRUTHFULQA_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(tqa_passed_df)} | Failed: {len(tqa_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

multiple_choice/validation-00000-of-0000(…):   0%|          | 0.00/271k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

[skip] Already complete (817 rows): truthfulQA_test_results.csv

Passed: 322 | Failed: 495


### 2.2 MMLU

In [6]:
mmlu_mc = load_dataset("cais/mmlu", "all", split="test")

kc_mmlu_df, mmlu_passed_df, mmlu_failed_df = run_knowledge_check_mmlu(
    mmlu_mc, None, tokenizer, DEVICE, MMLU_KC_PATH, CHECKPOINT_EVERY
)
print(f"\nPassed: {len(mmlu_passed_df)} | Failed: {len(mmlu_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

[skip] Already complete (14042 rows): mmlu_test_results.csv

Passed: 4794 | Failed: 9248


## Part 3: Factual Response Generation and Result Judge
### 3.1 TruthfulQA
#### 3.1.1 Response Generation

In [7]:
# tqa_resp_df = run_factual_generation(
#     tqa_passed_df, tqa_failed_df, model, tokenizer, DEVICE,
#     NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
#     TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY,
#     DO_SAMPLE,
# )
# print(tqa_resp_df["config"].value_counts().to_string())

#### 3.1.1b TruthfulQA — vLLM Response Generation

In [8]:
tqa_resp_df = run_factual_generation_vllm(
    tqa_passed_df, tqa_failed_df, llm, tokenizer,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    TRUTHFULQA_RESPONSES_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
)
print(tqa_resp_df["config"].value_counts().to_string())

Starting fresh: 1139 rows across 3 configs
Config A: 322 rows to generate.


Config A:   0%|          | 0/7 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/22 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Done. Total rows: 1139
config
B    495
A    322
C    322
config
B    495
A    322
C    322


#### 3.1.2 Claude Haiku Batch Judge

In [ ]:
tqa_haiku_df = run_judge_anthropic(
    tqa_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_TQA_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_TQA_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

Saved 6834 requests → judge_truthfulQA_requests.jsonl


[2026-05-07 20:14:34] INFO _client.py:1038: HTTP Request: POST https://api.anthropic.com/v1/messages/batches "HTTP/1.1 200 OK"
[2026-05-07 20:14:35] INFO _client.py:1038: HTTP Request: GET https://api.anthropic.com/v1/messages/batches/msgbatch_01EnHQ1FzPeEESoaWFUXo17J "HTTP/1.1 200 OK"


Submitted batch 1/1: msgbatch_01EnHQ1FzPeEESoaWFUXo17J (6834 requests)
[msgbatch_01EnHQ1FzPeEESoaWFUXo17J] in_progress | succeeded=0  processing=6834  errored=0
Waiting 180s... (1 batch(es) still running)


### 3.2 MMLU Response Generation and Result Judge
#### 3.2.1 Response Generation

In [ ]:
# mmlu_resp_df = run_factual_generation(
#     mmlu_passed_df, mmlu_failed_df, model, tokenizer, DEVICE,
#     NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
#     MMLU_RESPONSES_PATH, CHECKPOINT_EVERY,
#     DO_SAMPLE,
# )
# print(mmlu_resp_df["config"].value_counts().to_string())

In [ ]:
# output_path = Path(MMLU_RESPONSES_PATH)
# total_shards = 2

# shard_paths = [
#     output_path.parent / (output_path.stem + f"_shard{i}" + output_path.suffix)
#     for i in range(total_shards)
# ]

# mmlu_resp_df = pd.concat([pd.read_csv(p) for p in shard_paths], ignore_index=True)
# before = len(mmlu_resp_df)
# mmlu_resp_df = mmlu_resp_df.drop_duplicates(subset=["question", "config"]).reset_index(drop=True)
# print(f"Merged: {before} rows → {len(mmlu_resp_df)} after dedup")
# print(mmlu_resp_df["config"].value_counts().to_string())
# mmlu_resp_df.to_csv(output_path, index=False)
# print(f"Saved to {output_path}")

#### 3.2.1b MMLU — vLLM Response Generation

In [ ]:
mmlu_resp_df = run_factual_generation_vllm(
    mmlu_passed_df, mmlu_failed_df, llm, tokenizer,
    NEUTRAL_SYSTEM, FACTUAL_DECEPTION_SCENARIO,
    MMLU_RESPONSES_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
)
print(mmlu_resp_df["config"].value_counts().to_string())

#### 3.2.2 Claude Haiku Batch Judge

In [ ]:
mmlu_haiku_df = run_judge_anthropic(
    mmlu_resp_df,
    model=JUDGE_CLAUDE_HAIKU_MODEL,
    n_votes=VOTES_PER_MODEL,
    output_path=JUDGE_CLAUDE_HAIKU_MMLU_PATH,
    state_path=JUDGE_CLAUDE_HAIKU_MMLU_STATE,
    batch_dir=JUDGE_CLAUDE_HAIKU_BATCH_DIR,
)

### 3.3 Judge Result Compile

In [ ]:
if TRUTHFULQA_FULL_PATH.exists() and MMLU_FULL_PATH.exists():
    tqa_full  = pd.read_csv(TRUTHFULQA_FULL_PATH)
    mmlu_full = pd.read_csv(MMLU_FULL_PATH)
    print(f"Loaded tqa_full  ({len(tqa_full)} rows)")
    print(f"Loaded mmlu_full ({len(mmlu_full)} rows)")
else:
    tqa_votes  = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_TQA_PATH,
        vote_cols=VOTE_COLS,
    )
    mmlu_votes = aggregate_judge_votes(
        JUDGE_CLAUDE_HAIKU_MMLU_PATH,
        vote_cols=VOTE_COLS,
    )
    tqa_full  = build_full(tqa_votes,  tqa_resp_df)
    mmlu_full = build_full(mmlu_votes, mmlu_resp_df)
    tqa_full.to_csv(TRUTHFULQA_FULL_PATH,  index=False)
    mmlu_full.to_csv(MMLU_FULL_PATH, index=False)
    print(f"Saved tqa_full  ({len(tqa_full)} rows) → {TRUTHFULQA_FULL_PATH.name}")
    print(f"Saved mmlu_full ({len(mmlu_full)} rows) → {MMLU_FULL_PATH.name}")

print_threshold_summary(tqa_full,  "TruthfulQA")
print_threshold_summary(mmlu_full, "MMLU")

## Part 4: Scenario Response Generation

In [ ]:
# from utils.analysis import prepare_gemma4_thinking_dataset
# 
# gemma4_df = prepare_gemma4_thinking_dataset(DECEPTION_DATASET_PATH)

In [ ]:
# scenario_resp_df = run_scenario_generation(
#     deception_df, model, tokenizer, DEVICE,
#     SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY,
#     DO_SAMPLE,
# )
# print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
# print(scenario_resp_df.head(2))

### Part 4b: Scenario Response Generation — vLLM

In [ ]:
scenario_resp_df = run_scenario_generation_vllm(
    deception_df, llm, tokenizer,
    SCENARIO_RESPONSES_PATH, SCENARIO_RAW_PATH, CHECKPOINT_EVERY, DO_SAMPLE,
)
print(f"\nColumns: {scenario_resp_df.columns.tolist()}")
print(scenario_resp_df.head(2))

## Part 5: Build Probe Dataset and Extract Activations
### 5.1 Build Probe Dataset

In [ ]:
probe_dataset = build_probe_dataset(
    tqa_full, mmlu_full, scenario_resp_df, PROBE_DATASET_PATH,
)

In [ ]:
from utils.analysis import split_thinking_responses
probe_dataset_split = split_thinking_responses(
    probe_dataset,
    save_path=DATA_DIR / "probe_dataset_split.csv"
)

### 5.2 Extract Activations

In [ ]:
# Activation extraction requires the HuggingFace model (for hidden state hooks).
# vLLM does not expose per-layer hidden states — load HF model separately.
# Qwen3-4B ≈ 8GB bfloat16; if vLLM is still in memory and OOM occurs:
#   del llm; torch.cuda.empty_cache()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HF_READ_TOKEN,
)
model.eval()
cfg = getattr(model.config, "text_config", model.config)
N_LAYERS   = getattr(cfg, "num_hidden_layers", getattr(cfg, "num_layers", None))
HIDDEN_DIM = getattr(cfg, "hidden_size",       getattr(cfg, "d_model",    None))
print(f"HF model loaded: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}")

In [ ]:
_df = probe_dataset_split.copy()
_df["response"] = _df["response_answer"]

activations_arr, labels_arr = run_extract_activations(
    _df, model, tokenizer, DEVICE,
    ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN, CHECKPOINT_EVERY,
)
print(f"Label counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

### 5.2b Extract Activations from Split Answers

In [ ]:
# _df = probe_dataset_split.copy()
# _df["response"] = _df["response_answer"]

# activations_arr, labels_arr = run_extract_activations(
#     _df, model, tokenizer, DEVICE,
#     ACTIVATIONS_PATH, LABELS_PATH, ACTIVATIONS_CHECKPOINT_PATH,
#     HF_ACTIVATIONS_REPO, HF_TOKEN, CHECKPOINT_EVERY,
# )
# print(f"\nLabel counts: { {k: int((labels_arr == v).sum()) for k, v in LABEL_MAP.items()} }")

## Part 6: Probe Training and Evaluation
### 6.1 Setup

In [ ]:
labels_str = np.array([{v: k for k, v in LABEL_MAP.items()}[i] for i in labels_arr])

k_selection_df = select_pca_k(
    activations_arr, labels_str, PCA_K_VALUES, PCA_K_SELECTION_PATH,
)

In [ ]:
acts_reduced = run_pca_reduction(
    activations_arr, PCA_K,
    ACTIVATIONS_PCA_PATH, PCA_COMPONENTS_PATH, PCA_VARIANCE_PATH,
    HF_ACTIVATIONS_REPO, HF_READ_TOKEN,
)

### 6.2 Baseline: Binary Classifier

In [ ]:
results_binary_c1 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=1.0,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C1_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C1.pkl",
)
results_binary_c01 = probe_all_layers_binary(
    acts_reduced, labels_str,
    C=0.1,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=BINARY_C01_PATH,
    checkpoint_path=BINARY_DIR / "checkpoint_binary_C01.pkl",
)

In [ ]:
plot_auroc(
    [(results_binary_c1, "C=1.0"), (results_binary_c01, "C=0.1")],
    BINARY_DIR / "figures" / "auroc.png",
    title="Binary Probe AUROC per Layer (truth vs deception)",
)

### 6.3 Approach 1: Direct 3-Way LR Classifier

In [ ]:
results_3way_lr = probe_all_layers(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=TWAY_LR_PATH,
    checkpoint_path=TWAY_LR_DIR / "checkpoint_3way_lr.pkl",
)

In [ ]:
plot_macro_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "macro_f1.png", title="3-Way LR: Macro F1 per Layer")
plot_perclass_f1(results_3way_lr, TWAY_LR_DIR / "figures" / "perclass_f1.png", title="3-Way LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_lr, TWAY_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="LR ")

### 6.4 Approach 2: Direct 3-Way MLP Classifier

In [ ]:
results_3way_mlp = probe_all_layers_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=TWAY_MLP_PATH,
    checkpoint_path=TWAY_MLP_DIR / "checkpoint_3way_mlp.pkl",
)

In [ ]:
plot_macro_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "macro_f1.png", title="3-Way MLP: Macro F1 per Layer")
plot_perclass_f1(results_3way_mlp, TWAY_MLP_DIR / "figures" / "perclass_f1.png", title="3-Way MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_3way_mlp, TWAY_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="MLP ")
plot_macro_f1(
    [(results_3way_lr, "LR"), (results_3way_mlp, "MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_lr_vs_mlp.png",
    title="3-Way Probe: LR vs MLP Macro F1",
)

### 6.5 Approach 3: 2-Stage LR Classifier

In [ ]:
results_cascaded_lr = probe_all_layers_cascaded(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    output_path=CASCADED_LR_PATH,
    checkpoint_path=CASCADED_LR_DIR / "checkpoint_cascaded_lr.pkl",
)

In [ ]:
plot_macro_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "macro_f1.png", title="Cascaded LR: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "perclass_f1.png", title="Cascaded LR: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_lr, CASCADED_LR_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded LR ")

### 6.6 Approach 4: 2-Stage MLP Classifier

In [ ]:
results_cascaded_mlp = probe_all_layers_cascaded_mlp(
    acts_reduced, labels_str,
    n_splits=N_SPLITS, max_iter=MAX_ITER, random_state=RANDOM_STATE,
    hidden_layer_sizes=MLP_HIDDEN_LAYER_SIZES,
    output_path=CASCADED_MLP_PATH,
    checkpoint_path=CASCADED_MLP_DIR / "checkpoint_cascaded_mlp.pkl",
)

In [ ]:
plot_macro_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "macro_f1.png", title="Cascaded MLP: Macro F1 per Layer")
plot_perclass_f1(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "perclass_f1.png", title="Cascaded MLP: Per-Class F1 per Layer")
plot_top_confusion_matrices(results_cascaded_mlp, CASCADED_MLP_DIR / "figures" / "top5_cm.png", n_top=5, title_prefix="Cascaded MLP ")
plot_macro_f1(
    [(results_cascaded_lr, "Cascaded LR"), (results_cascaded_mlp, "Cascaded MLP")],
    OUTPUT_DIR / "figures" / "macro_f1_cascaded_lr_vs_mlp.png",
    title="Cascaded Probe: LR vs MLP Macro F1",
)

## Part 7: Model Comparison

In [ ]:
# Load all probe results from CSV (safe to run after kernel restart)
r_lr    = pd.read_csv(TWAY_LR_PATH)
r_mlp   = pd.read_csv(TWAY_MLP_PATH)
r_clr   = pd.read_csv(CASCADED_LR_PATH)
r_cmlp  = pd.read_csv(CASCADED_MLP_PATH)
r_bin1  = pd.read_csv(BINARY_C1_PATH)
r_bin01 = pd.read_csv(BINARY_C01_PATH)

PROBE_RESULTS = [
    (r_lr,   "3-Way LR"),
    (r_mlp,  "3-Way MLP"),
    (r_clr,  "Cascaded LR"),
    (r_cmlp, "Cascaded MLP"),
]
SUMMARY_DIR = OUTPUT_DIR / "summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
print("Loaded all probe results.")

In [ ]:
layers = r_lr["layer"].values

# ── Table 1: Macro F1 per layer ───────────────────────────────────────────────
t1 = pd.DataFrame({"layer": layers})
for df, name in PROBE_RESULTS:
    t1[name] = df["f1_macro"].values
t1.to_csv(SUMMARY_DIR / "summary_macro_f1.csv", index=False)
print(t1.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
for df, name in PROBE_RESULTS:
    ax.plot(layers, df["f1_macro"], marker="o", markersize=3, label=name)
ax.set_xlabel("Layer"); ax.set_ylabel("Macro F1")
ax.set_title("Macro F1 per Layer — All Probes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "macro_f1_all_probes.png", dpi=150); plt.close(fig)
print("Saved macro_f1_all_probes.png")

# ── Tables 2/3/4: Per-class F1 per layer ─────────────────────────────────────
for cls in ["truth", "honest_mistake", "deception"]:
    t = pd.DataFrame({"layer": layers})
    for df, name in PROBE_RESULTS:
        t[name] = df[f"f1_{cls}"].values
    t.to_csv(SUMMARY_DIR / f"summary_f1_{cls}.csv", index=False)
    print(f"\n── {cls} ──")
    print(t.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    for df, name in PROBE_RESULTS:
        ax.plot(layers, df[f"f1_{cls}"], marker="o", markersize=3, label=name)
    ax.set_xlabel("Layer"); ax.set_ylabel(f"F1 ({cls})")
    ax.set_title(f"{cls} F1 per Layer — All Probes")
    ax.legend(); ax.grid(True, alpha=0.3)
    fig.tight_layout(); fig.savefig(SUMMARY_DIR / f"f1_{cls}_all_probes.png", dpi=150); plt.close(fig)
    print(f"Saved f1_{cls}_all_probes.png")

# ── Table 5: AUROC per layer (binary baseline) ────────────────────────────────
t5 = pd.DataFrame({
    "layer":       r_bin1["layer"].values,
    "Binary C=1.0": r_bin1["auroc"].values,
    "Binary C=0.1": r_bin01["auroc"].values,
})
t5.to_csv(SUMMARY_DIR / "summary_auroc_binary.csv", index=False)
print("\n── Binary AUROC ──")
print(t5.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(r_bin1["layer"], r_bin1["auroc"], marker="o", markersize=3, label="C=1.0")
ax.plot(r_bin01["layer"], r_bin01["auroc"], marker="o", markersize=3, label="C=0.1")
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("Binary Probe AUROC per Layer (truth vs deception)")
ax.set_ylim(0.5, 1.0); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig(SUMMARY_DIR / "auroc_binary.png", dpi=150); plt.close(fig)
print("Saved auroc_binary.png")

In [ ]:
# ── Upload all large files to HuggingFace Hub ────────────────────────────────
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi()
outputs_root = Path("outputs")

# Collect gitignored large files to preview
files_to_upload = []
for pattern in ["**/*.npy", "**/*.npz"]:
    files_to_upload.extend(sorted(outputs_root.glob(pattern)))

print(f"Found {len(files_to_upload)} files to upload:")
for p in files_to_upload:
    print(f"  {p.as_posix()}")

# Upload entire outputs/ tree, preserving directory structure in the repo.
# Files land at e.g. outputs/qwen2.5-7b-instruct/.../activations.npy
# so different model runs never collide.
print(f"\nUploading to {HF_ACTIVATIONS_REPO} ...")
api.upload_folder(
    folder_path=str(outputs_root),
    path_in_repo="outputs",
    repo_id=HF_ACTIVATIONS_REPO,
    repo_type="dataset",
    token=HF_WRITE_TOKEN,
    allow_patterns=["**/*.npy", "**/*.npz"],
)
print("Upload complete.")